# 🍕 Fine-Tune ViT for Food Recognition

This notebook fine-tunes `google/vit-base-patch16-224` (Vision Transformer) on the **Food-101** dataset for accurate food recognition.

## Setup Instructions
1. **Enable GPU**: Go to `Runtime → Change Runtime Type → GPU (T4)`
2. **Run all cells**: `Runtime → Run All`
3. **Training time**: ~2-3 hours on T4 GPU
4. **Result**: Model pushed to your Hugging Face account

---

## 1️⃣ Install Dependencies & Check GPU

In [ ]:
# Install required packages
!pip install -q torch torchvision transformers datasets evaluate accelerate huggingface-hub scikit-learn matplotlib seaborn tqdm

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change Runtime Type → GPU")
    print("   Training without GPU will be extremely slow.")

## 2️⃣ Login to Hugging Face

You need a Hugging Face account to push the trained model.

1. Create a free account at [huggingface.co](https://huggingface.co)
2. Create an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
   - Token type: **Write** (so we can push the model)
3. Paste it below when prompted

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3️⃣ Configuration

Edit these settings if you want to adjust training. Defaults work well for Food-101.

In [ ]:
# ============================================================
# Configuration — edit these if needed
# ============================================================

MODEL_NAME = "google/vit-base-patch16-224"   # Pre-trained ViT model
DATASET_NAME = "ethz/food101"                # Food-101 dataset (101K images)

# Where to push on Hugging Face (change YOUR_USERNAME)
HF_MODEL_REPO = "food-vit-fine-tuned"        # Will be: YOUR_USERNAME/food-vit-fine-tuned

# Training hyperparameters
NUM_EPOCHS = 5               # 5 epochs is a good balance
BATCH_SIZE = 32              # Reduce to 16 if you get OOM errors
LEARNING_RATE = 2e-5         # Standard fine-tuning LR
WEIGHT_DECAY = 0.01          # L2 regularization
WARMUP_RATIO = 0.1           # Warmup 10% of training

# Freeze strategy
UNFREEZE_LAST_N_BLOCKS = 2   # Fine-tune only last 2 transformer blocks + classifier head

print("✅ Configuration set!")

## 4️⃣ Load Food-101 Dataset

In [ ]:
from datasets import load_dataset

print("📥 Loading Food-101 dataset (this may take a few minutes)...")
dataset = load_dataset(DATASET_NAME)

# Get class information
class_names = dataset['train'].features['label'].names
num_classes = len(class_names)
id2label = {i: name.replace('_', ' ').title() for i, name in enumerate(class_names)}
label2id = {v: k for k, v in id2label.items()}

print(f"\n✅ Dataset loaded!")
print(f"   Training images: {len(dataset['train']):,}")
print(f"   Validation images: {len(dataset['validation']):,}")
print(f"   Number of food classes: {num_classes}")
print(f"\n📝 Sample classes: {', '.join(list(id2label.values())[:10])}...")

### Preview sample images

In [ ]:
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Food-101 Sample Images', fontsize=16, fontweight='bold')

for ax in axes.flat:
    idx = random.randint(0, len(dataset['train']) - 1)
    sample = dataset['train'][idx]
    ax.imshow(sample['image'])
    ax.set_title(id2label[sample['label']], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5️⃣ Prepare Image Processor & Transforms

In [ ]:
from transformers import ViTImageProcessor
from torchvision import transforms

# Load the image processor for ViT
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
image_size = processor.size['height']  # 224

print(f"Image size: {image_size}x{image_size}")

# Training transforms with data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
])

# Evaluation transforms (no augmentation)
eval_transforms = transforms.Compose([
    transforms.Resize(image_size + 32),
    transforms.CenterCrop(image_size),
])

def preprocess_train(examples):
    images = [train_transforms(img.convert('RGB')) for img in examples['image']]
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

def preprocess_eval(examples):
    images = [eval_transforms(img.convert('RGB')) for img in examples['image']]
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

# Apply transforms
print("📦 Applying transforms to dataset...")
train_dataset = dataset['train'].with_transform(preprocess_train)
eval_dataset = dataset['validation'].with_transform(preprocess_eval)
print("✅ Transforms applied!")

## 6️⃣ Load & Configure ViT Model

In [ ]:
from transformers import ViTForImageClassification

print(f"🧠 Loading {MODEL_NAME}...")

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # We're replacing the classifier head
)

# ============================================================
# Freeze early layers (transfer learning)
# We only fine-tune the classifier head + last N encoder blocks
# This is faster and prevents catastrophic forgetting
# ============================================================

# Freeze embeddings
for param in model.vit.embeddings.parameters():
    param.requires_grad = False

# Freeze all encoder blocks except the last N
total_blocks = len(model.vit.encoder.layer)
freeze_until = total_blocks - UNFREEZE_LAST_N_BLOCKS

for i, layer in enumerate(model.vit.encoder.layer):
    if i < freeze_until:
        for param in layer.parameters():
            param.requires_grad = False

# Print summary
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded!")
print(f"   Frozen: {freeze_until}/{total_blocks} encoder blocks")
print(f"   Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"   Model size: {total * 4 / 1e9:.2f} GB (float32)")

## 7️⃣ Train the Model! 🏋️

This will take approximately **2-3 hours** on a T4 GPU. You can monitor progress in the output below.

**Tip**: Don't close this tab while training! Colab may disconnect if idle for too long.

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

# Metrics
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Training arguments
training_args = TrainingArguments(
    output_dir='./checkpoints/food101_vit',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),    # Mixed precision for faster training
    logging_steps=50,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=2,           # Colab has limited CPU cores
    save_total_limit=2,                 # Keep only 2 best checkpoints
    push_to_hub=False,                  # We'll push manually after training
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# Start training!
print("🚀 Starting training...")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Device: {'CUDA (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")
print()

train_result = trainer.train()

## 8️⃣ Evaluate Results

In [ ]:
# Final evaluation on the full validation set
print("📊 Evaluating on test set...")
eval_results = trainer.evaluate()

print("\n" + "=" * 50)
print("🎉 TRAINING COMPLETE!")
print("=" * 50)
print(f"   Test Accuracy: {eval_results.get('eval_accuracy', 0):.4f} ({eval_results.get('eval_accuracy', 0)*100:.1f}%)")
print(f"   Test Loss:     {eval_results.get('eval_loss', 0):.4f}")
print("=" * 50)

if eval_results.get('eval_accuracy', 0) >= 0.85:
    print("\n✅ Great accuracy! This model is ready for deployment.")
elif eval_results.get('eval_accuracy', 0) >= 0.75:
    print("\n⚠️ Decent accuracy. Consider training for more epochs or with a larger batch size.")
else:
    print("\n❌ Low accuracy. Try training for more epochs or unfreezing more layers.")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

history = trainer.state.log_history

# Extract epoch-level metrics
eval_epochs = [h['epoch'] for h in history if 'eval_accuracy' in h]
eval_accs = [h['eval_accuracy'] for h in history if 'eval_accuracy' in h]
eval_losses = [h['eval_loss'] for h in history if 'eval_loss' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(eval_epochs, eval_accs, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Validation Accuracy per Epoch')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])
for i, acc in enumerate(eval_accs):
    ax1.annotate(f'{acc:.1%}', (eval_epochs[i], acc), textcoords="offset points", xytext=(0, 10), ha='center')

# Loss plot
ax2.plot(eval_epochs, eval_losses, 'r-o', linewidth=2, markersize=8)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Validation Loss per Epoch')
ax2.grid(True, alpha=0.3)

plt.suptitle('Food-101 ViT Fine-Tuning Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9️⃣ Test with Sample Images

Let's test the model on some images from the validation set to see how it performs.

In [ ]:
import torch
import matplotlib.pyplot as plt
import random

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Model Predictions on Validation Images', fontsize=14, fontweight='bold')

for ax in axes.flat:
    idx = random.randint(0, len(dataset['validation']) - 1)
    sample = dataset['validation'][idx]
    image = sample['image'].convert('RGB')
    true_label = id2label[sample['label']]

    # Run inference
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_id = probs.argmax(-1).item()
    pred_label = id2label[pred_id]
    confidence = probs[0][pred_id].item()

    # Display
    ax.imshow(image)
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {pred_label}\n({confidence:.0%})\nTrue: {true_label}', 
                 fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 🔟 Push Model to Hugging Face Hub

This uploads the trained model to your Hugging Face account so it can be used via the Inference API.

After this step, your model will be available at:
```
https://api-inference.huggingface.co/models/YOUR_USERNAME/food-vit-fine-tuned
```

In [ ]:
from huggingface_hub import HfApi

# Get username
api = HfApi()
user_info = api.whoami()
username = user_info['name']
repo_id = f"{username}/{HF_MODEL_REPO}"

print(f"📤 Pushing model to: {repo_id}")
print("   This may take a few minutes...\n")

# Save model locally first
model.save_pretrained('./final_model')
processor.save_pretrained('./final_model')

# Push to Hub
model.push_to_hub(repo_id, private=True)
processor.push_to_hub(repo_id, private=True)

print(f"\n✅ Model pushed successfully!")
print(f"\n📋 Copy these values to your app's .env file:")
print(f"   EXPO_PUBLIC_HF_API_URL=https://api-inference.huggingface.co/models/{repo_id}")
print(f"\n🔗 View your model: https://huggingface.co/{repo_id}")
print(f"\n💡 To get an API token for your app:")
print(f"   Go to: https://huggingface.co/settings/tokens")
print(f"   Create a 'Read' token and add it as EXPO_PUBLIC_HF_API_TOKEN in your .env")

## 1️⃣1️⃣ Quick Inference Test via API

Test that the Hugging Face Inference API works with your model.

In [ ]:
import requests
import io

# Test the HF Inference API
API_URL = f"https://api-inference.huggingface.co/models/{repo_id}"
headers = {"Authorization": f"Bearer {api.token}"}

# Use a random validation image
test_sample = dataset['validation'][random.randint(0, len(dataset['validation']) - 1)]
test_image = test_sample['image'].convert('RGB')
true_label = id2label[test_sample['label']]

# Convert to bytes
buf = io.BytesIO()
test_image.save(buf, format='JPEG')
image_bytes = buf.getvalue()

print(f"🔍 Testing inference API...")
print(f"   True label: {true_label}")
print(f"   API URL: {API_URL}")

response = requests.post(API_URL, headers=headers, data=image_bytes)

if response.status_code == 200:
    results = response.json()
    print(f"\n✅ API Response:")
    for r in results[:5]:
        print(f"   {r['label']}: {r['score']:.1%}")
elif response.status_code == 503:
    print("\n⏳ Model is loading... This is normal for the first request.")
    print("   Wait 1-2 minutes and try again.")
else:
    print(f"\n❌ Error: {response.status_code}")
    print(f"   {response.text}")

---

## ✅ Done! Next Steps

1. **Copy the model URL and token** from the output above to your app's `.env` file:
   ```
   EXPO_PUBLIC_HF_API_URL=https://api-inference.huggingface.co/models/YOUR_USERNAME/food-vit-fine-tuned
   EXPO_PUBLIC_HF_API_TOKEN=hf_YOUR_READ_TOKEN
   ```

2. **The model is now accessible** via the Hugging Face Inference API. Your app's scan feature will call this API to recognize food!

3. **The first API call may be slow** (~30s) as the model loads. Subsequent calls will be fast (~1-2s).

4. **Free tier limits**: HF Inference API has generous free limits (thousands of requests/month). For a computing project, this is more than enough.